# Hoeffding's D Feature Selection

This notebook calculates Hoeffding's D statistic between each feature and the binary target (loan_status) to assess **nonlinear dependence**.

Unlike correlation or mutual information, Hoeffding's D can detect complex relationships, making it useful for evaluating variables in credit risk models where monotonicity is not guaranteed.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from scipy.stats import rankdata

import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## Hoeffding’s D Function


This custom function implements Hoeffding’s D calculation using ranked data, based on pairwise concordance and discordance. It's adapted from the original statistical definition and assumes continuous or ordinal inputs.


In [2]:
def hoeffdingsD( x, y ):

    #N = size(x,1);
    N=x.shape
    #R = tiedrank( x );
    R=rankdata(x)
    #S = tiedrank( y );
    S=rankdata(y)

    #Q = zeros(N,1);
    Q=np.zeros(N[0])
    #parfor i = 1:N
    for i in range(0, N[0]):
        #Q[i] = 1 + sum( R < R[i] & S < S[i] );
        Q[i] = 1 + np.sum(np.bitwise_and(R<R[i] ,S<S[i]))
        #% and deal with cases where one or both values are ties, which contribute less
        #Q[i] = Q[i] + 1/4 * (sum( R == R[i] & S == S[i] ) - 1); #% both indices tie.  -1 because we know point i matches
        Q[i] = Q[i] + 1/4 * (np.sum(np.bitwise_and(np.isin(R,R[i]),np.isin(S,S[i])))-1)
        #Q[i] = Q[i] + 1/2 * sum( R == R[i] & S < S[i] ); #% one index ties.
        Q[i] = Q[i] + 1/2 * (np.sum(np.bitwise_and(np.isin(R,R[i]),S<S[i])))
        #Q[i] = Q[i] + 1/2 * sum( R < R[i] & S == S[i] ); #% one index ties.
        Q[i] = Q[i] + 1/2 * (np.sum(np.bitwise_and(R<R[i],np.isin(S,S[i]))))
    #D1 = sum( (Q-1).*(Q-2) );
    #D2 = sum( (R-1).*(R-2).*(S-1).*(S-2) );
    #D3 = sum( (R-2).*(S-2).*(Q-1) );
    D1 = np.sum( np.multiply((Q-1),(Q-2)) );
    D2 = np.sum( np.multiply(np.multiply((R-1),(R-2)),np.multiply((S-1),(S-2)) ) );
    D3 = np.sum( np.multiply(np.multiply((R-2),(S-2)),(Q-1)) );


    D = 30*((N[0]-2)*(N[0]-3)*D1 + D2 - 2*(N[0]-2)*D3) / (N[0]*(N[0]-1)*(N[0]-2)*(N[0]-3)*(N[0]-4));
    #p=(N[0]-1)*D*math.pow(math.pi, 4)/60+math.pow(math.pi, 4)/72

    return D

## Load Univariate Dataset

We use the univariate WOE/IV-prepared dataset (excluding IV scores) to evaluate feature-target dependence. Only features relevant for scorecard-style models are included.

In [3]:
data = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_test_0_datasetForUnivariateVSExceptIV.csv', index_col=[0])

In [4]:
data.head()

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,1.002263,63788.0,17,-51.803563,5000.0,33.130018,11.01,0.08,12.0,686,-105.711715,0
11,1.002263,13113.0,0,131.015335,4500.0,-25.457321,8.63,0.34,2.0,651,-105.711715,1
9010,0.369066,59603.0,0,-51.803563,8000.0,11.545480,14.96,0.13,2.0,570,-105.711715,1
7030,-1.125529,54919.0,0,-51.803563,6225.0,55.131699,11.54,0.11,4.0,706,914.813057,0
21143,1.002263,55000.0,7,77.011839,6000.0,11.545480,9.32,0.11,9.0,643,914.813057,0


In [5]:
data.shape

(32400, 12)

In [6]:
lstfeature = list(data.columns)

In [7]:
len(lstfeature)

12

In [8]:
lstfeature.remove('loan_status')

In [9]:
len(lstfeature)

11

In [10]:
variable = []
hoeff = []

## Loop Through Features

We loop through each predictor variable and calculate its Hoeffding's D value relative to loan_status. This allows us to rank features by nonlinear predictive strength.


In [11]:
for i in lstfeature:
    variable.append(i)
    hoeff.append(hoeffdingsD(data[i], data['loan_status']))

In [12]:
dict = {'variable': variable, 'hoeff': hoeff}
df = pd.DataFrame(dict)

## Interpretation of Hoeffding's D Results


Hoeffding’s D identified a small subset of features with measurable nonlinear dependence on loan_status.

- **previous_loan_defaults_on_file** stands out with the highest score (≈ 0.024), confirming its strong predictive potential, consistent with earlier XGBoost and IV findings.

- Moderate signal is observed in **loan_percent_income** and **loan_int_rate**, indicating they may capture important risk relationships not strictly linear.

- Most features have D values close to zero, and a few are slightly negative, suggesting little to no dependence with the target, these may be dropped or deprioritized in modeling.

> **Note:** Hoeffding's D values are often small in magnitude; what matters more is the relative ranking across features.

This test adds a nonparametric, monotonicity-agnostic lens to feature evaluation, complementing techniques like IV and model-based importance scores

In [13]:
df

,variable,hoeff
0,person_education,-0.000032
1,person_income,0.008884
2,person_emp_exp,0.000047
3,person_home_ownership,0.005867
4,loan_amnt,0.001310
5,loan_intent,0.002432
6,loan_int_rate,0.012947
7,loan_percent_income,0.013673
8,cb_person_cred_hist_length,0.000015
9,credit_score,-0.000020


## Sort and Export Results

The resulting feature importance scores are sorted in descending order and exported to CSV for further analysis or comparison with other selection methods like IV or XGBoost.


In [14]:
df = df.sort_values(by = ['hoeff'], ascending = False).reset_index().drop(columns = ['index'])

In [15]:
df.to_csv('Internal_hoeff.csv')